# Выбор кластерного разбиения по ограниченной внешней информации


## 1. Постановка задачи

Для набора объектов $X=\{x_i\}_{i=1}^n$ без использования меток строится фиксированный пул разбиений $\mathcal P=\{P_1,\ldots,P_M\}$. Из резерва объектов раскрываются метки подмножества $L$, $|L|=m$. Метки отдельной проверочной части $T$ скрыты от правил выбора. Признаки всех объектов доступны заранее: используется трансдуктивная постановка.

Внешняя оценка на доступной разметке — $A_L(P)=\operatorname{ARI}(y_L,P_L)$. Внутренняя оценка $I(P)$ рассчитывается по признакам и разбиению. Проверочное качество $Q_T(P)=\operatorname{ARI}(y_T,P_T)$ вычисляется после выбора.

Внешний лидер $P_{\mathrm{ari\_max}}=\arg\max_P A_L(P)$. Для кандидата $P$ определяем отставание $d_P=A_L(P_{\mathrm{ari\_max}})-A_L(P)$. На одинаковых бутстрепных перевыборках размеченных объектов оцениваем стандартное отклонение парной разности $\widehat\sigma_P$. Кластеризации остаются фиксированными.

Адаптивное правило допускает $\mathcal E_c=\{P:d_P\le c\widehat\sigma_P+10^{-12}\}$ и выбирает $P_{\mathrm{silh\_max}}=\arg\max_{P\in\mathcal E_c}I(P)$. Допуск является эвристикой. Отсутствующие в разметке классы не появляются при бутстрепе.

Цель — установить условия пользы и вреда внутреннего индекса. Основные факторы: бюджет меток, покрытие классов, геометрия данных и состав пула. Все методы получают одинаковые метки. Измеряются проверочный ARI, потеря $R_T(P)=\max_{P'\in\mathcal P}Q_T(P')-Q_T(P)$, частота смены лидера и последствия переключений. Повторы разметки одного набора характеризуют вариативность внутри задачи.


### Правила выбора и измерение результата

| Метод | Правило |
|---|---|
| M0 | Максимум внутреннего индекса |
| M1 | Максимум ARI на доступных метках |
| M2 | Внутренний индекс разрешает равенства внешней оценки |
| M3 | Фиксированная смесь нормированных оценок |
| M4 | Смесь с весом, настроенным на отдельной development-серии |
| M5 | Внутренний максимум при фиксированном допуске по ARI |
| M6 | Внутренний максимум при бутстрепном допуске |
| M7 | Максимальная частота бутстрепных побед с разделением голосов при равенствах |


## 2. Установка проекта и библиотек


In [ ]:
import sys
import subprocess
import time
import json
import hashlib
import os
import re
import site
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile, ZIP_DEFLATED
from uuid import uuid4
from importlib import invalidate_caches
from google.colab import files
from IPython.display import display, Image

gpu_status = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
nvcc_status = subprocess.run(["nvcc", "--version"], capture_output=True, text=True, check=True)
match = re.search(r"release\s+(12|13)\.", nvcc_status.stdout)
if not match:
    raise RuntimeError("Нужен CUDA toolkit 12.x или 13.x; проверьте вывод nvcc --version")
cuda_major = match.group(1)
print("CUDA toolkit:", cuda_major, "|", gpu_status.stdout.splitlines()[0])

WORKSPACE = Path("/content") / ("clusterization_run_" + uuid4().hex[:8])
ROOT = WORKSPACE / "clusterization"
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Загрузите один ZIP-архив проекта")
archive_name, archive_bytes = next(iter(uploaded.items()))
with ZipFile(BytesIO(archive_bytes)) as archive:
    if "clusterization/pyproject.toml" not in archive.namelist():
        raise ValueError("В архиве нет проекта clusterization")
    archive.extractall(WORKSPACE)

install_result = subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--extra-index-url", "https://pypi.nvidia.com",
    "-e", str(ROOT),
    f"cuml-cu{cuda_major}==26.6.0",
    f"cupy-cuda{cuda_major}x",
], capture_output=True, text=True)
if install_result.returncode:
    print("Последние строки pip stdout:\\n", install_result.stdout[-6000:])
    print("Последние строки pip stderr:\\n", install_result.stderr[-6000:])
    raise RuntimeError("Установка проекта и GPU-зависимостей не завершилась")
print("pip: установка завершена")

library_dirs = []
for package_dir in site.getsitepackages():
    base = Path(package_dir)
    for package in ("libcuml", "libraft", "libcudf", "librmm", "pylibraft"):
        directory = base / package / "lib64"
        if directory.is_dir():
            library_dirs.append(str(directory))
    library_dirs.extend(str(directory) for directory in base.glob("nvidia/*/lib")
                        if directory.is_dir())
existing_library_path = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = ":".join(
    dict.fromkeys(library_dirs + ([existing_library_path] if existing_library_path else []))
)

source_dir = str(ROOT / "src")
if source_dir in sys.path:
    sys.path.remove(source_dir)
sys.path.insert(0, source_dir)
for module_name in list(sys.modules):
    if module_name == "clusterization" or module_name.startswith("clusterization."):
        del sys.modules[module_name]
invalidate_caches()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from clusterization.config import load_configs, validate
from clusterization.candidates import load_pool
from clusterization.data import prepared_dir
from clusterization.metrics import ari_grid

### Исполняемые эксперименты


In [ ]:
OPTIONAL_RUNS = set()

CONFIG_FILES = sorted((ROOT / "configs/experiments").glob("*.nip"))
configs = {}
catalog = []
for path in CONFIG_FILES:
    variants = load_configs(path)
    for variant in variants:
        validate(variant)
    first = variants[0]
    configs[first["name"]] = first
    catalog.append({
        "конфиг": first["name"], "вариантов": len(variants),
        "backend": first["compute"]["backend"], "фаза": first["phase"],
        "данные": first["dataset"]["kind"],
        "набор": first["dataset"].get("name", first["dataset"].get("geometry")),
        "сценарии": ", ".join(first["selection"]["scenarios"]),
        "бюджеты": str(first["selection"]["budgets"]),
        "повторы": first["selection"]["repeats"],
        "bootstrap": first["selection"]["bootstrap_draws"],
    })
unknown = OPTIONAL_RUNS - set(configs)
if unknown:
    raise ValueError(f"Неизвестные конфиги: {sorted(unknown)}")
display(pd.DataFrame(catalog))
print("Выбраны дополнительные опыты:", sorted(OPTIONAL_RUNS) or "нет")

### Общее выполнение и файлы результатов


In [ ]:
def run_script(filename, arguments):
    command = [sys.executable, str(ROOT / "scripts" / filename), *arguments]
    print("START:", filename, " ".join(arguments), flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    print(f"END: {filename}; exit={code}; {time.perf_counter() - started:.1f} с")
    if code:
        raise subprocess.CalledProcessError(code, command)

def execute_experiment(name):
    variants = load_configs(ROOT / "configs/experiments" / f"{name}.nip")
    config = variants[0]
    if config["phase"] == "development":
        raise ValueError("Для development используйте run_optional ниже")
    config_path = ROOT / "configs/experiments" / f"{name}.nip"
    for stage in ("prepare_data.py", "build_candidates.py", "run_experiment.py"):
        run_script(stage, ["--config", str(config_path)])
    report = ROOT / "reports" / (name + "_" + uuid4().hex[:8])
    run_script("summarize_results.py",
               ["--results", str(ROOT / "results" / name), "--out", str(report)])
    result = {
        "name": name, "config": config, "report": report,
        "summary": pd.read_csv(report / "per_dataset.csv"),
        "selections": pd.read_csv(report / "selections.csv"),
        "paired": pd.read_csv(report / "paired_contrasts.csv"),
    }
    assert not result["summary"].empty and not result["selections"].empty
    print("Строк решений:", len(result["selections"]), "| отчёт:", report)
    return result

def show_summary(result, limit=30):
    columns = [name for name in ["series", "dataset", "cell", "scenario", "budget",
               "method", "mean_regret", "mean_ari", "near_optimal_rate",
               "mean_coverage", "mean_eligible", "repeats"]
               if name in result["summary"].columns]
    table = result["summary"][columns].sort_values(
        [name for name in ["series", "dataset", "scenario", "budget", "mean_regret"]
         if name in columns]).reset_index(drop=True)
    display(table.head(limit))
    print("Показано", min(limit, len(table)), "из", len(table), "строк")

### Общие графики


In [ ]:
def show_figure(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)

def plot_metric(result, metric, ylabel):
    frame = result["summary"]
    scenarios = sorted(frame.scenario.unique())
    fig, axes = plt.subplots(1, len(scenarios), figsize=(7 * len(scenarios), 4),
                             squeeze=False)
    for axis, scenario in zip(axes[0], scenarios):
        part = frame[frame.scenario == scenario]
        for method, values in part.groupby("method"):
            values = values.sort_values("budget")
            axis.plot(values.budget, values[metric], marker="o", label=method)
        axis.set(title=scenario, xlabel="Размеченных объектов", ylabel=ylabel)
        axis.grid(alpha=0.3)
        axis.legend(ncol=4, fontsize=8)
    show_figure(fig, result["report"] / f"{metric}_by_budget.png")

def plot_coverage(result):
    frame = result["summary"]
    fig, axis = plt.subplots(figsize=(7, 4))
    for scenario, values in frame[frame.method == "M0"].groupby("scenario"):
        values = values.sort_values("budget")
        axis.plot(values.budget, values.mean_coverage, marker="o", label=scenario)
    axis.set(title="Покрытие классов разметкой", xlabel="Размеченных объектов",
             ylabel="Доля известных классов", ylim=(0, 1.05))
    axis.grid(alpha=0.3)
    axis.legend()
    show_figure(fig, result["report"] / "class_coverage.png")

def paired_gains(result, baseline="M0", selected="M6"):
    frame = result["selections"].pivot(
        index=["scenario", "budget", "repeat"], columns="method", values="regret")
    return (frame[baseline] - frame[selected]).rename("gain").reset_index()

def plot_gain(result, baseline="M0", selected="M6"):
    gains = paired_gains(result, baseline, selected)
    fig, axis = plt.subplots(figsize=(8, 4))
    for scenario, values in gains.groupby("scenario"):
        axis.scatter(values.budget, values.gain, alpha=0.45,
                     label=f"{scenario}: повторы")
        mean = values.groupby("budget", as_index=False).gain.mean()
        axis.plot(mean.budget, mean.gain, marker="o", linewidth=2,
                  label=f"{scenario}: среднее")
    axis.axhline(0, color="black", linewidth=1)
    axis.set(title=f"Парный выигрыш {selected} относительно {baseline}",
             xlabel="Размеченных объектов",
             ylabel=f"Regret {baseline} − regret {selected}")
    axis.grid(alpha=0.3)
    axis.legend(fontsize=8)
    show_figure(fig, result["report"] / f"gain_{selected}_vs_{baseline}.png")
    display(gains.groupby(["scenario", "budget"], as_index=False).agg(
        mean_gain=("gain", "mean"), repeats=("gain", "size")))

## 3. Опыт 0 — пилот на синтетических полумесяцах


In [ ]:
pilot = execute_experiment("gpu_pilot")
show_summary(pilot)

### Рисунки пилота


In [ ]:
plot_metric(pilot, "mean_regret", "Средний внешний regret ↓")
plot_metric(pilot, "mean_ari", "Средний тестовый ARI ↑")
plot_gain(pilot)

### Кандидаты: внутренний индекс против внешнего ARI


In [ ]:
partitions, pool = load_pool(ROOT, pilot["config"])
with np.load(prepared_dir(ROOT, pilot["config"]) / "labels.npz") as saved:
    hidden_y = saved["y"]
report_manifest = json.loads((pilot["report"] / "manifest.json").read_text())
run_dir = Path(report_manifest["sources"][0]["run"])
with np.load(run_dir / "split.npz") as saved:
    test_indices = saved["test"]
candidate_ari = ari_grid(hidden_y[test_indices], partitions[:, test_indices])[0]
candidate_rows = pd.DataFrame([
    {"algorithm": item["aliases"][0]["algorithm"], "k": item["k"],
     "coverage": item["coverage"], "silhouette": item["silhouette"],
     "test_ari": float(candidate_ari[i])}
    for i, item in enumerate(pool["candidates"])
])
display(candidate_rows.sort_values("test_ari", ascending=False))
fig, axis = plt.subplots(figsize=(7, 5))
for algorithm, part in candidate_rows.groupby("algorithm"):
    axis.scatter(part.silhouette, part.test_ari, label=algorithm, s=60)
axis.set(title="Кандидаты пилота: геометрия и метки",
         xlabel="Внутренний силуэт", ylabel="Тестовый ARI")
axis.grid(alpha=0.3)
axis.legend()
show_figure(fig, pilot["report"] / "candidate_silhouette_vs_test_ari.png")

## 4. Опыт 1 — Digits: влияние неполного покрытия классов


In [ ]:
digits_sil = execute_experiment("gpu_digits_light")
show_summary(digits_sil)

### Рисунки и результат Digits


In [ ]:
plot_coverage(digits_sil)
plot_metric(digits_sil, "mean_regret", "Средний внешний regret ↓")
plot_metric(digits_sil, "mean_ari", "Средний тестовый ARI ↑")
plot_gain(digits_sil)
display(digits_sil["paired"].query("baseline in ['M0', 'M1', 'M5']")[
    ["scenario", "budget", "baseline", "mean_gain_M6",
     "ci_low", "ci_high", "n_repeats"]
].sort_values(["scenario", "budget", "baseline"]))

## 5. Опыт 2 — Digits: силуэт против Davies–Bouldin


In [ ]:
digits_db = execute_experiment("gpu_digits_db_light")
show_summary(digits_db)

### Парное сравнение индексов


In [ ]:
keys = ["dataset", "scenario", "budget", "repeat", "method"]
comparison = digits_sil["selections"].merge(
    digits_db["selections"], on=keys, how="inner",
    suffixes=("_sil", "_db"), validate="one_to_one")
assert len(comparison) == len(digits_sil["selections"]) == len(digits_db["selections"])
for method in ("M1", "M7"):
    control = comparison[comparison.method == method]
    assert (control.selected_id_sil == control.selected_id_db).all(), method
comparison["delta_regret"] = comparison.regret_db - comparison.regret_sil
comparison_summary = comparison.groupby(
    ["scenario", "budget", "method"], as_index=False).agg(
        mean_delta=("delta_regret", "mean"), repeats=("repeat", "nunique"))
display(comparison_summary.sort_values(["scenario", "budget", "mean_delta"]))
fig, axes = plt.subplots(1, 2, figsize=(13, 4), squeeze=False)
for axis, scenario in zip(axes[0], sorted(comparison_summary.scenario.unique())):
    part = comparison_summary[comparison_summary.scenario == scenario]
    for method, values in part.groupby("method"):
        axis.plot(values.budget, values.mean_delta, marker="o", label=method)
    axis.axhline(0, color="black", linewidth=1)
    axis.set(title=scenario, xlabel="Размеченных объектов",
             ylabel="Regret DB − regret silhouette")
    axis.grid(alpha=0.3)
    axis.legend(ncol=4, fontsize=8)
show_figure(fig, digits_db["report"] / "db_vs_silhouette_paired.png")

## 6. Опыт 3 — Breast Cancer: двухклассовый контроль


In [ ]:
binary = execute_experiment("gpu_breast_cancer_light")
show_summary(binary)

### Рисунки и результат двухклассового контроля


In [ ]:
plot_coverage(binary)
plot_metric(binary, "mean_regret", "Средний внешний regret ↓")
plot_metric(binary, "mean_ari", "Средний тестовый ARI ↑")
plot_gain(binary)

## 7. Дополнительные опыты: порядок и зависимости


In [ ]:
selected_text = any(configs[name]["dataset"]["kind"] == "text"
                    for name in OPTIONAL_RUNS)
if selected_text:
    print("Устанавливаются текстовые зависимости проекта...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    f"{ROOT}[text]"], check=True)
else:
    print("Текстовые зависимости сейчас не нужны")

### Общий помощник для длинных серий


In [ ]:
def run_optional(name):
    config = configs[name]
    path = ROOT / "configs/experiments" / f"{name}.nip"
    backend = config["compute"]["backend"]
    frozen = ROOT / "artifacts" / (
        "frozen_mixture_cuda.json" if backend == "cuda" else "frozen_mixture.json")

    if config["phase"] == "development":
        if not frozen.exists():
            run_script("prepare_data.py", ["--config", str(path)])
            run_script("build_candidates.py", ["--config", str(path)])
            run_script("tune_mixture.py", ["--config", str(path)])
        artifact = json.loads(frozen.read_text())
        scores = pd.DataFrame([
            {"lambda": float(weight), "mean_regret": value}
            for weight, value in artifact["scores"].items()
        ]).sort_values("lambda")
        print("Замороженный вес M4:", artifact["lambda"], "| файл:", frozen)
        display(scores)
        ax = scores.plot(x="lambda", y="mean_regret", marker="o",
                         title=f"Настройка M4: {name}", figsize=(6, 4))
        ax.set_ylabel("Средняя внешняя потеря")
        display(ax.figure)
        plt.close(ax.figure)
        return {"name": name, "phase": "development",
                "frozen": frozen, "scores": scores}

    if "M4" in config["selection"]["methods"] and not frozen.exists():
        prerequisite = "gpu_development" if backend == "cuda" else "development"
        raise RuntimeError(f"Сначала запустите {prerequisite}: нужен {frozen}")

    if config["dataset"]["kind"] == "text":
        run_script("download_data.py", ["--config", str(path)])
        run_script("download_models.py", ["--config", str(path)])
        run_script("prepare_data.py", ["--config", str(path)])
        run_script("encode_texts.py", ["--config", str(path)])
    elif config["dataset"]["kind"] == "tabular":
        run_script("download_data.py", ["--config", str(path)])
    return execute_experiment(name)

def show_optional(result):
    if result.get("phase") == "development":
        return
    show_summary(result, limit=20)
    images = sorted(result["report"].glob("regret-*.png"))
    print("PNG-графиков в отчёте:", len(images))
    if images:
        print("Пример:", images[0])
        display(Image(filename=str(images[0])))

additional_results = {}

### 7.1. Обучение веса M4


In [ ]:
if "development" in OPTIONAL_RUNS:
    additional_results["development"] = run_optional("development")
    show_optional(additional_results["development"])
else:
    print("Пропущен development: добавьте имя в OPTIONAL_RUNS")

### 7.2. Отдельная настройка M4


In [ ]:
if "gpu_development" in OPTIONAL_RUNS:
    additional_results["gpu_development"] = run_optional("gpu_development")
    show_optional(additional_results["gpu_development"])
else:
    print("Пропущен gpu_development: добавьте имя в OPTIONAL_RUNS")

### 7.3. CPU-пилот на простой синтетике


In [ ]:
if "pilot" in OPTIONAL_RUNS:
    additional_results["pilot"] = run_optional("pilot")
    show_optional(additional_results["pilot"])
else:
    print("Пропущен pilot: добавьте имя в OPTIONAL_RUNS")

### 7.4. Подтверждающая синтетика на CPU


In [ ]:
if "synthetic" in OPTIONAL_RUNS:
    additional_results["synthetic"] = run_optional("synthetic")
    show_optional(additional_results["synthetic"])
else:
    print("Пропущен synthetic: добавьте имя в OPTIONAL_RUNS")

### 7.5. Подтверждающая синтетика на GPU


In [ ]:
if "gpu_synthetic" in OPTIONAL_RUNS:
    additional_results["gpu_synthetic"] = run_optional("gpu_synthetic")
    show_optional(additional_results["gpu_synthetic"])
else:
    print("Пропущен gpu_synthetic: добавьте имя в OPTIONAL_RUNS")

### 7.6. Табличные данные на CPU: Digits и Dry Bean


In [ ]:
if "tabular" in OPTIONAL_RUNS:
    additional_results["tabular"] = run_optional("tabular")
    show_optional(additional_results["tabular"])
else:
    print("Пропущен tabular: добавьте имя в OPTIONAL_RUNS")

### 7.7. Табличные данные на GPU: Digits и Dry Bean


In [ ]:
if "gpu_tabular" in OPTIONAL_RUNS:
    additional_results["gpu_tabular"] = run_optional("gpu_tabular")
    show_optional(additional_results["gpu_tabular"])
else:
    print("Пропущен gpu_tabular: добавьте имя в OPTIONAL_RUNS")

### 7.8. Два класса на CPU: Breast Cancer


In [ ]:
if "tabular_binary" in OPTIONAL_RUNS:
    additional_results["tabular_binary"] = run_optional("tabular_binary")
    show_optional(additional_results["tabular_binary"])
else:
    print("Пропущен tabular_binary: добавьте имя в OPTIONAL_RUNS")

### 7.9. Два класса на GPU: Breast Cancer


In [ ]:
if "gpu_tabular_binary" in OPTIONAL_RUNS:
    additional_results["gpu_tabular_binary"] = run_optional("gpu_tabular_binary")
    show_optional(additional_results["gpu_tabular_binary"])
else:
    print("Пропущен gpu_tabular_binary: добавьте имя в OPTIONAL_RUNS")

### 7.10. Текстовый перенос на CPU


In [ ]:
if "text" in OPTIONAL_RUNS:
    additional_results["text"] = run_optional("text")
    show_optional(additional_results["text"])
else:
    print("Пропущен text: добавьте имя в OPTIONAL_RUNS")

### 7.11. Текстовый перенос на GPU


In [ ]:
if "gpu_text" in OPTIONAL_RUNS:
    additional_results["gpu_text"] = run_optional("gpu_text")
    show_optional(additional_results["gpu_text"])
else:
    print("Пропущен gpu_text: добавьте имя в OPTIONAL_RUNS")

### 7.13. Смена внутреннего индекса на Davies–Bouldin


In [ ]:
if "text_db" in OPTIONAL_RUNS:
    additional_results["text_db"] = run_optional("text_db")
    show_optional(additional_results["text_db"])
else:
    print("Пропущен text_db: добавьте имя в OPTIONAL_RUNS")

### 7.14. Резервный выбор при отсутствии пригодной внешней оценки


In [ ]:
if "text_fallback" in OPTIONAL_RUNS:
    additional_results["text_fallback"] = run_optional("text_fallback")
    show_optional(additional_results["text_fallback"])
else:
    print("Пропущен text_fallback: добавьте имя в OPTIONAL_RUNS")

## 8. Cовместный результат


In [ ]:
experiments = [pilot, digits_sil, digits_db, binary] + [
    value for value in additional_results.values()
    if value.get("phase") != "development"
]
overview = pd.concat([
    item["summary"].assign(experiment=item["name"])
    for item in experiments
], ignore_index=True)
focus = overview[overview["method"].isin(["M0", "M1", "M6"])]
columns = [name for name in [
    "experiment", "series", "dataset", "cell", "scenario", "budget", "method",
    "mean_regret", "mean_ari", "near_optimal_rate", "mean_coverage", "repeats"
] if name in focus.columns]
display(focus[columns].head(100))
print("Показано", min(100, len(focus)), "из", len(focus), "строк")
print("Полные отчёты:")
for item in experiments:
    print(item["name"], "→", item["report"])

## 9. Выгрузка результатов


In [ ]:
bundle = Path("/content/clusterization_research_results.zip")
with ZipFile(bundle, "w", ZIP_DEFLATED) as output:
    for path in CONFIG_FILES:
        output.write(path, path.relative_to(ROOT))
    for item in experiments:
        for folder in (ROOT / "results" / item["name"], item["report"]):
            if folder.exists():
                for path in folder.rglob("*"):
                    if path.is_file():
                        output.write(path, path.relative_to(ROOT))
    for item in additional_results.values():
        if item.get("phase") == "development":
            output.write(item["frozen"], item["frozen"].relative_to(ROOT))
print("Архив готов:", bundle)
files.download(str(bundle))